# MM-Fit: RandomForest vs 1D-CNN (Klassifikation)

Dieses Notebook erstellt Sliding-Window-Segmente aus dem MM-Fit Datensatz und vergleicht eine RandomForest-Baseline
mit einem 1D-CNN.


## 1) Setup und Reproduzierbarkeit

Neuer versuch:
Windowing auf 250 und Step auf 10 setzten
 

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

import tensorflow as tf
from tensorflow.keras import layers, models

# Reproduzierbarkeit
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Pfade und Konfiguration
ROOT = "mm-fit"
TARGETS = ["pushups", "squats", "situps"]
label_map = {"pushups": 0, "squats": 1, "situps": 2}
inv_label_map = {v: k for k, v in label_map.items()}

SENSOR = "sw_r"
WIN = 250
STEP = 10
TEST_SIZE = 0.20
VAL_SIZE_OF_REST = 0.20

print("OK - Setup loaded")


OK - Setup loaded


/Users/kacharino/myProjects/Bachelor/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


**Datensatz**
- Root: `mm-fit/` mit Sessions `w00`–`w20`
- Pro Session: `*_sw_r_acc.npy`, `*_sw_r_gyr.npy`, `*_labels.csv`
- Labels: `start, end, reps, exercise` (nur pushups/squats/situps)


## 2) Sessions und dynamischer Split (train/val/test, gruppiert nach Session)


Train/Val/Test wird zufällig über Sessions gesplittet. Dadurch bleiben überlappende Fenster innerhalb einer Session im selben Split (kein Leakage).


In [2]:
def list_sessions(root=ROOT):
    return sorted([
        d for d in os.listdir(root)
        if d.startswith("w") and os.path.isdir(os.path.join(root, d))
    ])

sessions = list_sessions(ROOT)

outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)
idx_train_val, idx_test = next(outer_split.split(sessions, groups=sessions))

train_val_sessions = [sessions[i] for i in idx_train_val]
test_sessions = [sessions[i] for i in idx_test]

inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=VAL_SIZE_OF_REST,
    random_state=RANDOM_SEED,
)
idx_train, idx_val = next(inner_split.split(train_val_sessions, groups=train_val_sessions))

train_sessions = [train_val_sessions[i] for i in idx_train]
val_sessions = [train_val_sessions[i] for i in idx_val]

print("Anzahl Sessions:", len(sessions))
print("Train sessions (n=%d):" % len(train_sessions), train_sessions)
print("Val sessions   (n=%d):" % len(val_sessions), val_sessions)
print("Test sessions  (n=%d):" % len(test_sessions), test_sessions)


Anzahl Sessions: 21
Train sessions (n=12): ['w04', 'w05', 'w06', 'w09', 'w10', 'w11', 'w12', 'w13', 'w14', 'w16', 'w18', 'w20']
Val sessions   (n=4): ['w02', 'w03', 'w07', 'w19']
Test sessions  (n=5): ['w00', 'w01', 'w08', 'w15', 'w17']


## 3) Laden einer Session (ACC + GYR, x/y/z)


In [3]:
def load_one_session(wdir, sensor=SENSOR):
    base = os.path.join(ROOT, wdir)

    acc = np.load(os.path.join(base, f"{wdir}_{sensor}_acc.npy"))
    gyr = np.load(os.path.join(base, f"{wdir}_{sensor}_gyr.npy"))

    # x, y, z (Spalten 1 bis 3)
    acc_xyz = acc[:, 1:4].astype(np.float32)
    gyr_xyz = gyr[:, 1:4].astype(np.float32)

    labels = pd.read_csv(
        os.path.join(base, f"{wdir}_labels.csv"),
        header=None,
        names=["start", "end", "reps", "exercise"]
    )

    labels = labels[labels["exercise"].isin(TARGETS)].reset_index(drop=True)
    return acc_xyz, gyr_xyz, labels


## 4) Windowing (win=128, step=64)


In [4]:
def make_windows(acc_xyz, gyr_xyz, labels_df, win=WIN, step=STEP):
    X, y = [], []
    for _, row in labels_df.iterrows():
        start, end = int(row["start"]), int(row["end"])
        lab = label_map[row["exercise"]]

        for i in range(start, end - win + 1, step):
            w_acc = acc_xyz[i:i+win]
            w_gyr = gyr_xyz[i:i+win]
            X.append(np.hstack([w_acc, w_gyr]))  # (win, 6)
            y.append(lab)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


## 5) Dataset bauen

In [5]:
def build_from_session(wdir):
    acc_xyz, gyr_xyz, labels = load_one_session(wdir)
    X, y = make_windows(acc_xyz, gyr_xyz, labels)
    return X, y


def build_dataset(session_list, split_name="split"):
    X_all, y_all = [], []
    for wdir in session_list:
        X, y = build_from_session(wdir)
        if len(X) == 0:
            print("Skip (no windows):", wdir)
            continue
        X_all.append(X)
        y_all.append(y)
        print(f"[{split_name}]", wdir, "->", X.shape, np.unique(y, return_counts=True))

    if not X_all:
        raise ValueError(f"No windows found for split '{split_name}'.")

    return np.concatenate(X_all), np.concatenate(y_all)

X_train, y_train = build_dataset(train_sessions, split_name="train")
X_val, y_val = build_dataset(val_sessions, split_name="val")
X_test, y_test = build_dataset(test_sessions, split_name="test")

print("\nFINAL:")
print("X_train:", X_train.shape, "y_train:", y_train.shape, np.unique(y_train, return_counts=True))
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape,   np.unique(y_val, return_counts=True))
print("X_test :", X_test.shape,  "y_test :", y_test.shape,  np.unique(y_test, return_counts=True))


[train] w04 -> (417, 250, 6) (array([0, 1, 2]), array([ 64, 150, 203]))
[train] w05 -> (224, 250, 6) (array([0, 1, 2]), array([55, 90, 79]))
[train] w06 -> (282, 250, 6) (array([0, 1, 2]), array([ 78,  99, 105]))
[train] w09 -> (426, 250, 6) (array([0, 1, 2]), array([ 92, 137, 197]))
[train] w10 -> (297, 250, 6) (array([0, 1, 2]), array([ 81, 100, 116]))
[train] w11 -> (456, 250, 6) (array([0, 1, 2]), array([ 90, 147, 219]))
[train] w12 -> (244, 250, 6) (array([0, 1, 2]), array([ 39,  88, 117]))
[train] w13 -> (272, 250, 6) (array([0, 1, 2]), array([ 38,  87, 147]))
[train] w14 -> (243, 250, 6) (array([0, 1, 2]), array([ 63,  80, 100]))
[train] w16 -> (403, 250, 6) (array([0, 1, 2]), array([141, 103, 159]))
[train] w18 -> (452, 250, 6) (array([0, 1, 2]), array([ 84, 148, 220]))
[train] w20 -> (449, 250, 6) (array([0, 1, 2]), array([115, 170, 164]))
[val] w02 -> (370, 250, 6) (array([0, 1, 2]), array([ 59, 124, 187]))
[val] w03 -> (248, 250, 6) (array([0, 1, 2]), array([ 68,  73, 107]))

## 6) Normalisierung (train-basiert)


In [6]:
mu = X_train.mean(axis=(0, 1), keepdims=True)
sigma = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_n = (X_train - mu) / sigma
X_val_n   = (X_val   - mu) / sigma
X_test_n  = (X_test  - mu) / sigma

print("Normalized shapes:", X_train_n.shape, X_val_n.shape, X_test_n.shape)
print("Train mean ~", X_train_n.mean(), "Train std ~", X_train_n.std())


Normalized shapes: (4165, 250, 6) (1455, 250, 6) (1515, 250, 6)
Train mean ~ -0.30600205 Train std ~ 0.9546336


In [7]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print("class_weights:", class_weights)


class_weights: {np.int64(0): np.float64(1.4769503546099292), np.int64(1): np.float64(0.9923755063140338), np.int64(2): np.float64(0.7603139832055494)}


## 7) RandomForest 


In [8]:
X_train_rf = X_train_n.reshape(X_train_n.shape[0], -1)
X_test_rf  = X_test_n.reshape(X_test_n.shape[0], -1)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_rf, y_train)

y_pred_rf = rf.predict(X_test_rf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=[inv_label_map[i] for i in sorted(inv_label_map)]
))


Random Forest Accuracy: 0.4508250825082508

Confusion Matrix:
[[ 29 163 103]
 [ 28 280 199]
 [ 45 294 374]]

Classification Report:
              precision    recall  f1-score   support

     pushups       0.28      0.10      0.15       295
      squats       0.38      0.55      0.45       507
      situps       0.55      0.52      0.54       713

    accuracy                           0.45      1515
   macro avg       0.41      0.39      0.38      1515
weighted avg       0.44      0.45      0.43      1515



## 8) 1D-CNN (Conv1D + BatchNorm + Dropout)


In [9]:
def build_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv1D(64, 7, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn = build_cnn(input_shape=(WIN, X_train_n.shape[-1]), num_classes=len(label_map))
cnn.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 250, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 250, 64)        │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 250, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 250, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 125, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 125, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 125, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 62, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 62, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 62, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 62, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 177,475 (693.26 KB)

 Trainable params: 176,579 (689.76 KB)

 Non-trainable params: 896 (3.50 KB)

**Hinweis:** Train/Val/Test ist session-basiert. Das vermeidet Leakage durch stark überlappende Fenster bei `WIN=250` und `STEP=10`.


In [10]:
# Data augmentation: jitter + scaling
# (applied on-the-fly to training windows)
def augment(x, y):
    noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=0.02)
    scale = tf.random.uniform([tf.shape(x)[0], 1, 1], 0.9, 1.1)
    x = x * scale + noise
    return x, y

batch_size = 32

X_tr, y_tr = X_train_n, y_train
X_val_split, y_val_split = X_val_n, y_val

train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
train_ds = train_ds.shuffle(min(8192, len(X_tr)), seed=RANDOM_SEED).batch(batch_size).map(augment).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val_split, y_val_split))
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5
    )
]

history = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=80,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/80
131/131 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.5104 - loss: 1.0145 - val_accuracy: 0.3313 - val_loss: 1.0764 - learning_rate: 0.0010
Epoch 2/80
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6079 - loss: 0.8294 - val_accuracy: 0.4914 - val_loss: 1.1902 - learning_rate: 0.0010
Epoch 3/80
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6377 - loss: 0.7822 - val_accuracy: 0.4289 - val_loss: 1.7808 - learning_rate: 0.0010
Epoch 4/80
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6663 - loss: 0.7326 - val_accuracy: 0.4440 - val_loss: 2.0875 - learning_rate: 0.0010
Epoch 5/80
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6900 - loss: 0.7024 - val_accuracy: 0.5093 - val_loss: 2.0778 - learning_rate: 0.0010
Epoch 6/80
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7172 - loss: 0.6347 - val_accuracy: 0.4227 - val_loss: 2.5434 - learning_rate: 5.0000e-04
Epoch 7/80
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7371 - loss

## 8) Evaluation (Accuracy, Confusion Matrix)



In [11]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _HAS_SEABORN = True
except Exception:
    _HAS_SEABORN = False

# CNN Evaluation
probs = cnn.predict(X_test_n, verbose=0)
y_pred_cnn = probs.argmax(axis=1)

print("CNN Accuracy:", accuracy_score(y_test, y_pred_cnn))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_cnn))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_cnn,
    target_names=[inv_label_map[i] for i in sorted(inv_label_map)],
    zero_division=0
))


CNN Accuracy: 0.5392739273927393

Confusion Matrix:
[[ 98  98  99]
 [232 173 102]
 [ 76  91 546]]

Classification Report:
              precision    recall  f1-score   support

     pushups       0.24      0.33      0.28       295
      squats       0.48      0.34      0.40       507
      situps       0.73      0.77      0.75       713

    accuracy                           0.54      1515
   macro avg       0.48      0.48      0.48      1515
weighted avg       0.55      0.54      0.54      1515

